In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets scikit-learn
print("✅ Step 0: Dependencies installed")


Looking in indexes: https://download.pytorch.org/whl/cu118



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Step 0: Dependencies installed



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch

print("✅ Step 1: GPU Check")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training will be very slow.")


✅ Step 1: GPU Check
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Load CSV
df = pd.read_csv(r"C:\Users\sonu\Desktop\big_data\project\ML\merged_dataset.csv")

# Drop rows with missing text
df = df.dropna(subset=['selftext']).reset_index(drop=True)

# Encode labels
if 'label_enc' not in df.columns:
    le = LabelEncoder()
    df['label_enc'] = le.fit_transform(df['label'])
    print("✅ Labels encoded:", dict(zip(le.classes_, le.transform(le.classes_))))

# Train / Validation / Test Split
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['selftext'], df['label_enc'], test_size=0.2, stratify=df['label_enc'], random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

# Optional: print class distribution
print("✅ Step 2: Dataset split")
print("Train samples:", len(train_texts), "Validation samples:", len(val_texts), "Test samples:", len(test_texts))
print("Train class distribution:\n", train_labels.value_counts())


✅ Labels encoded: {'anx': np.int64(0), 'dep': np.int64(1), 'lone': np.int64(2), 'mh': np.int64(3), 'sw': np.int64(4)}
✅ Step 2: Dataset split
Train samples: 1416971 Validation samples: 177121 Test samples: 177122
Train class distribution:
 label_enc
1    490210
4    356796
3    231995
0    214911
2    123059
Name: count, dtype: int64


In [7]:
from datasets import Dataset
from transformers import RobertaTokenizer

print("⏳ Step 3: Loading tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Convert pandas DataFrame splits into HuggingFace Datasets
print("⏳ Step 3: Converting to HuggingFace Dataset format...")
train_dataset = Dataset.from_pandas(pd.DataFrame({"text": train_texts, "labels": train_labels}))
val_dataset = Dataset.from_pandas(pd.DataFrame({"text": val_texts, "labels": val_labels}))
test_dataset = Dataset.from_pandas(pd.DataFrame({"text": test_texts, "labels": test_labels}))

# Tokenization function
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization (no num_proc for Windows/Jupyter)
print("⏳ Step 3: Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Step 3: Tokenization complete")


⏳ Step 3: Loading tokenizer...
⏳ Step 3: Converting to HuggingFace Dataset format...
⏳ Step 3: Tokenizing datasets...


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 177122/177122 [13:45<00:00, 214.49 examples/s]

✅ Step 3: Tokenization complete


In [12]:
# import torch

# print("⏳ Saving tokenized datasets...")

# torch.save(train_dataset, "train_dataset.pt")
# torch.save(val_dataset, "val_dataset.pt")
# torch.save(test_dataset, "test_dataset.pt")

# print("✅ Tokenized datasets saved successfully!")


⏳ Saving tokenized datasets...
✅ Tokenized datasets saved successfully!


In [8]:
print("⏳ Saving tokenized datasets properly using HuggingFace method...")

train_dataset.save_to_disk("train_dataset")
val_dataset.save_to_disk("val_dataset")
test_dataset.save_to_disk("test_dataset")

print("✅ Tokenized datasets saved successfully!")


⏳ Saving tokenized datasets properly using HuggingFace method...


Saving the dataset (1/1 shards): 100%|██████████████████████████████████████████████████████████████████████| 177122/177122 [00:00<00:00, 235318.41 examples/s]

✅ Tokenized datasets saved successfully!


In [1]:
from datasets import load_from_disk

print("⏳ Loading tokenized datasets from disk...")

train_dataset = load_from_disk("train_dataset")
val_dataset = load_from_disk("val_dataset")
test_dataset = load_from_disk("test_dataset")

print("✅ Tokenized datasets loaded successfully!")


C:\Users\sonu\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading tokenized datasets from disk...
✅ Tokenized datasets loaded successfully!


In [18]:
import accelerate
print(accelerate.__version__)  # should print 1.10.1

from transformers import is_accelerate_available
print(is_accelerate_available())  # should print True


1.10.1


ImportError: cannot import name 'is_accelerate_available' from 'transformers' (C:\Users\sonu\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\__init__.py)

In [4]:
from datasets import load_from_disk
from transformers import BertTokenizer

# Load raw text datasets (not the tokenized ones!)
raw_train = load_from_disk("raw_train")  
raw_val = load_from_disk("raw_val")
raw_test = load_from_disk("raw_test")

# Init BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = raw_train.map(tokenize_fn, batched=True)
val_dataset = raw_val.map(tokenize_fn, batched=True)
test_dataset = raw_test.map(tokenize_fn, batched=True)

# Save new tokenized datasets
train_dataset.save_to_disk("bert_train_dataset")
val_dataset.save_to_disk("bert_val_dataset")
test_dataset.save_to_disk("bert_test_dataset")


FileNotFoundError: Directory raw_train not found

In [2]:
import torch, torchvision, transformers, accelerate, datasets
print(torch.__version__, torchvision.__version__, transformers.__version__, accelerate.__version__, datasets.__version__)


2.2.0+cpu 0.17.0+cpu 4.56.2 0.27.0 4.1.1
